# Day 39 — Gradient boosting intro (XGBoost/LightGBM)
Objectives:
- Train a GBM model.
- Tune key params (learning_rate, n_estimators, max_depth).
- Overfitting/underfitting tradeoffs.

In [ ]:
from sklearn.datasets import load_breast_cancer
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
X,y = load_breast_cancer(return_X_y=True)
Xtr,Xte,ytr,yte = train_test_split(X,y,random_state=42)
xgb = XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=4, subsample=0.9, colsample_bytree=0.9, eval_metric='logloss', random_state=42)
xgb.fit(Xtr,ytr)
xgb.score(Xte,yte)


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — sequential boosting, learning-rate budgets, and validation control

### Mental model

Boosting builds an additive model one weak learner at a time. Each new
learner is chosen to reduce the current loss, so later trees focus on
mistakes left by earlier trees. The learning rate shrinks each tree's
contribution; the estimator count controls how many correction steps
are available.

XGBoost and LightGBM implement optimized, regularized variants with
different tree-growth and histogram strategies. Backend names do not
replace experiment design: use the same split, feature contract,
metric, compute budget, and early-stopping boundary when comparing
them.

### Read the API before running it

- **`n_estimators`:** sets the maximum boosting rounds; it interacts strongly with learning rate.
- **`learning_rate`:** shrinks each new tree, often requiring more rounds for similar fit.
- **validation/early stopping:** uses a training-external validation boundary to choose round count; the final test remains untouched.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — watch staged predictions reduce residual error

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** The squared-error loss and shallow trees are suitable for this smooth synthetic relationship.

In [ ]:
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor

rng = np.random.default_rng(3901)
X = np.linspace(-2, 2, 160).reshape(-1, 1)
y = X[:, 0] ** 2 + rng.normal(scale=0.15, size=X.shape[0])
model = GradientBoostingRegressor(
    n_estimators=40, learning_rate=0.08, max_depth=2, random_state=3901
).fit(X, y)
staged_mse = [
    np.mean((y - prediction) ** 2)
    for prediction in model.staged_predict(X)
]
print({"first": staged_mse[0], "last": staged_mse[-1]})
assert staged_mse[-1] < staged_mse[0]

**Expected observation:** Training loss falls as additive correction stages are added; this alone does not prove validation improvement.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — compare two learning-rate and round-count budgets

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Both candidates receive the same untouched validation rows and metric.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

X, y = load_breast_cancer(return_X_y=True)
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, stratify=y, random_state=3902
)
for rate, rounds in ((0.2, 30), (0.03, 200)):
    model = GradientBoostingClassifier(
        learning_rate=rate, n_estimators=rounds,
        max_depth=2, random_state=3902
    ).fit(X_train, y_train)
    auc = roc_auc_score(y_valid, model.predict_proba(X_valid)[:, 1])
    print({"learning_rate": rate, "rounds": rounds, "valid_auc": auc})

**Expected observation:** Different rate/round combinations can achieve similar validation performance, so one hyperparameter cannot be judged alone.

### Debugging and practice ramp

**Common mistake:** Tuning rounds against the final test set or assuming a lower training loss means a better model.

**Diagnostic:** Plot training and validation metric by round, record best iteration, compare wall time, and hold all other split/metric choices fixed.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define sequential boosting, learning-rate budgets, and validation control in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not compare XGBoost and LightGBM results produced from different preprocessing, row splits, metrics, or early-stopping data.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Tune `learning_rate` and `n_estimators` with a simple loop.

**Verify:** Practice 1 — sequential boosting, learning-rate budgets, and validation control — evaluate every (learning_rate, n_estimators) pair on one frozen training/validation split and scorer; print the complete score table, selected pair, fit count, and validation metric without consulting final-test labels.

2. Compare XGBoost with LightGBM if the `ml` dependency group is installed.

**Verify:** Practice 2 — sequential boosting, learning-rate budgets, and validation control — first print an explicit installed/skipped capability result for XGBoost and LightGBM; when both run, use identical rows, splits, metric, seed, and search budget and report validation score, best iteration, and elapsed time for each.

### Progressive hints

1. Use a small grid, hold `max_depth` constant, and evaluate all combinations
   with identical folds or an explicit validation set. Record runtime too.
2. Match the split, metric, approximate capacity, and random seed. For LightGBM,
   `num_leaves` plays a related—but not identical—complexity role to depth.

### Additional mastery practice

Control boosting capacity with honest validation, early stopping, and reproducible optional backends. Ranking, calibration, and runtime are separate outcomes.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

3. **Early-stopping design:** Create train, validation, and final test boundaries for early stopping. Explain why using the test set as the early-stopping evaluation set invalidates the final score.
   **Progressive hint:** The stopping iteration is a selected hyperparameter. Only the validation set may guide it; the test set remains untouched.

**Verify:** Early-stopping design — record disjoint train/early-stop-validation/final-test index hashes, best iteration selected without test labels, and one final-test score computed only after stopping is frozen.

4. **Calibration check:** Compare ROC AUC, log loss, and a reliability diagram for a boosting classifier. Construct an example where ranking is good but probability estimates are overconfident.
   **Progressive hint:** AUC depends on ordering; log loss and calibration depend on the numeric probabilities. Use a separate calibration boundary.

**Verify:** Calibration check — print ROC AUC, log loss, Brier score, and reliability-bin counts/mean prediction/event rate; include a constructed overconfident score transform that preserves ranking/AUC but worsens probability calibration.

5. **Portable backend contract:** Design a comparison helper that uses scikit-learn's HistGradientBoostingClassifier offline and adds XGBoost/LightGBM only when installed. It must report skipped backends explicitly.
   **Progressive hint:** Detect availability with importlib, keep the baseline unconditional, and never convert a missing optional package into a silent pass.

**Verify:** Portable backend contract — return a result row per backend with name, installed/skipped status, version, seed, fit time, and metric; assert the offline scikit-learn row always exists and missing optional imports never trigger downloads.

6. **Overfitting diagnosis:** Plot training and validation loss by boosting iteration and diagnose a curve where training loss falls continuously while validation loss starts rising.
   **Progressive hint:** The iteration at minimum validation loss is the candidate stopping point. Also test depth, minimum leaf support, subsampling, and learning rate.

**Verify:** Overfitting diagnosis — save train/validation loss by iteration, print the minimum-validation-loss iteration, and assert the diagnosed overfit region begins where validation rises while training continues falling.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 3 — Early-stopping design


# Practice 4 — Calibration check


# Practice 5 — Portable backend contract


# Practice 6 — Overfitting diagnosis
